# Azure ML — Triton Inference Server on a Managed Online Endpoint

This notebook deploys ONNX models to **NVIDIA Triton Inference Server** running on an Azure ML Managed Online Endpoint.

## Architecture

Azure ML online endpoints are built from four resources:

| Resource | Purpose |
|---|---|
| **Endpoint** | A stable HTTPS URL — just a placeholder, no compute attached. |
| **Model** | The model artifacts ([./models](cci:9://file:///Users/michal.wojdylak/projects/azureml-examples/sdk/python/endpoints/online/triton/single-model/models:0:0-0:0) dir) mounted into the container via `model_mount_path`. |
| **Environment** | A custom Docker image pushed to ACR. We need Triton **24.09+**, built from the [Dockerfile](cci:7://file:///Users/michal.wojdylak/projects/azureml-examples/sdk/python/endpoints/online/triton/single-model/Dockerfile:0:0-0:0) in this directory. |
| **Deployment** | The actual GPU VM (`STANDARD_NC4AS_T4_V3`) that pulls all the above together and starts serving. |

Adding a new model **requires a redeployment** — Triton loads the model repository at startup and does not watch for changes.

Inference uses `tritonclient` talking directly to the Triton HTTP v2 endpoint through Azure's SSL proxy. Each model is addressed by name.

## Prerequisites

- Azure subscription with an AML workspace and a linked ACR.
- The Triton image pushed to ACR:
  ```bash
  az acr build --registry <acr_name> --image tritonserver:24.09-py3 -f Dockerfile .

## 1. Imports

In [ ]:
import random

import gevent.ssl
import numpy as np
import tritonclient.http as tritonhttpclient
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    Environment,
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
    Model,
)
from azure.identity import DefaultAzureCredential
from PIL import Image
from transformers import AutoConfig, AutoImageProcessor

## 2. Configuration

In [ ]:
SUBSCRIPTION_ID = "SUBSCRIPTION_ID"
RESOURCE_GROUP = "RESOURCE_GROUP"
WORKSPACE_NAME = "WORKSPACE_NAME"
ACR_NAME = "ACR_NAME"

ENDPOINT_NAME = f"endpoint-{random.randint(0, 10000)}"

TRITON_IMAGE_TAG = "24.09-py3"
INSTANCE_TYPE = "STANDARD_NC4AS_T4_V3"
MODEL_LOCAL_PATH = "./models"
TEST_IMAGE_PATH = "./test_image.jpg"

MODELS = {
    "resnet50_onnx": {
        "hf_model": "microsoft/resnet-50",
        "input_name": "pixel_values",
        "output_name": "logits",
    },
    "convnext_tiny_onnx": {
        "hf_model": "facebook/convnext-tiny-224",
        "input_name": "pixel_values",
        "output_name": "logits",
    },
    "mobilenet_v2_onnx": {
        "hf_model": "google/mobilenet_v2_1.0_224",
        "input_name": "pixel_values",
        "output_name": "logits",
        "processor_kwargs": {"crop_size": {"height": 256, "width": 256}},
    },
}

## 3. Helpers

In [ ]:
def softmax(logits: np.ndarray) -> np.ndarray:
    exp = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
    return exp / np.sum(exp, axis=-1, keepdims=True)


def decode_top1(logits: np.ndarray, hf_model: str) -> tuple[str, float]:
    config = AutoConfig.from_pretrained(hf_model)
    probs = softmax(logits)
    pred_idx = int(np.argmax(probs, axis=-1)[0])
    score = float(probs[0, pred_idx])
    label = config.id2label.get(pred_idx, f"class_{pred_idx}")
    return label, score

## 4. Connect to Azure ML Workspace

In [ ]:
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION_ID,
    RESOURCE_GROUP,
    WORKSPACE_NAME,
)

## 5. Create Endpoint & Deployment

- **Endpoint** — empty placeholder (no GPU yet)
- **Environment** — custom Triton image from ACR
- **Model** — local model repo mounted into the container at `/triton`
- **Deployment** — GPU VM that merges endpoint + model + environment

In [ ]:
# ---- Endpoint (empty placeholder — no GPU yet) ----
endpoint = ManagedOnlineEndpoint(name=ENDPOINT_NAME, auth_mode="key")

# ---- Environment (custom Triton image from ACR) ----
triton_env = Environment(
    image=f"{ACR_NAME}.azurecr.io/tritonserver:{TRITON_IMAGE_TAG}",
    inference_config={
        "scoring_route": {"port": 8000, "path": "/"},
        "liveness_route": {"port": 8000, "path": "/v2/health/live"},
        "readiness_route": {"port": 8000, "path": "/v2/health/ready"},
    },
)

# ---- Model (local model repo mounted into the container) ----
model = Model(name="triton", path=MODEL_LOCAL_PATH, type="custom_model")

# ---- Deployment (GPU VM — merges endpoint + model + env) ----
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    instance_type=INSTANCE_TYPE,
    instance_count=1,
    environment=triton_env,
    model_mount_path="/triton",
)

In [ ]:
# 1. Create the endpoint (fast — no compute involved)
endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()

# 2. Create the deployment (provisions GPU VM — takes ~10 min)
ml_client.online_deployments.begin_create_or_update(deployment).result()

# 3. Route 100% traffic to the new deployment
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

## 6. Inference

We use `tritonclient` to talk directly to Triton's HTTP v2 protocol through Azure's SSL proxy. Each model is addressed by its Triton name.

In [ ]:
endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
scoring_uri = endpoint.scoring_uri
auth_key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key

# Strip the scheme — tritonclient expects host:port only
url = scoring_uri[8:]

triton_client = tritonhttpclient.InferenceServerClient(
    url=url,
    ssl=True,
    ssl_context_factory=gevent.ssl._create_default_https_context,
)

headers = {"Authorization": f"Bearer {auth_key}"}

health_ctx = triton_client.is_server_ready(headers=headers)
print(f"Is server ready - {health_ctx}")

In [ ]:
img = Image.open(TEST_IMAGE_PATH).convert("RGB")

for model_name_triton, cfg in MODELS.items():
    status_ctx = triton_client.is_model_ready(model_name_triton, "1", headers)
    print(f"Is {model_name_triton} ready - {status_ctx}")

    # Client-side preprocessing with HuggingFace
    processor_kwargs = cfg.get("processor_kwargs", {})
    processor = AutoImageProcessor.from_pretrained(cfg["hf_model"], **processor_kwargs)
    inputs = processor(images=img, return_tensors="np")
    pixel_values = inputs["pixel_values"].astype(np.float32)

    # Build Triton inference request via tritonclient
    input_tensor = tritonhttpclient.InferInput(
        cfg["input_name"], list(pixel_values.shape), "FP32"
    )
    input_tensor.set_data_from_numpy(pixel_values)

    result = triton_client.infer(
        model_name_triton,
        [input_tensor],
        outputs=[tritonhttpclient.InferRequestedOutput(cfg["output_name"])],
        headers=headers,
    )
    logits = result.as_numpy(cfg["output_name"])

    label, score = decode_top1(logits, cfg["hf_model"])
    print(f"{model_name_triton:<30s}  label={label:<40s}  score={score:.4f}")

## 7. Clean up resources

Run this section when you no longer need the endpoint.

In [ ]:
delete = False

In [ ]:
if delete:
    ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME)